# Headcount Forecasting & Workforce Planning

This notebook demonstrates the forecasting module of the People Analytics suite.

**What we cover:**
1. Historical headcount trends by department
2. Time series forecasting (Trend+Seasonal vs. Exponential Smoothing)
3. Model comparison and selection
4. Attrition risk prediction
5. What-if scenario planning

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (10, 5)

from src.forecasting.headcount_model import HeadcountForecaster, TrendSeasonalModel
from src.forecasting.attrition_model import AttritionModel
from src.forecasting.scenario_planner import ScenarioPlanner, Scenario

## 1. Historical Headcount Trends

In [ ]:
headcount = pd.read_csv('../data/headcount_history.csv')
employees = pd.read_csv('../data/employees.csv')
surveys = pd.read_csv('../data/survey_responses.csv')

print(f'Headcount records: {len(headcount):,}')
print(f'Departments: {headcount["department"].nunique()}')
print(f'Date range: {headcount["date"].min()} to {headcount["date"].max()}')
headcount.head()

In [ ]:
# Historical headcount by department
fig, ax = plt.subplots(figsize=(12, 5))
for dept in headcount['department'].unique():
    dept_data = headcount[headcount['department'] == dept].sort_values('date')
    ax.plot(range(len(dept_data)), dept_data['headcount'], label=dept, linewidth=1.5)
ax.set_xlabel('Month Index')
ax.set_ylabel('Headcount')
ax.set_title('Historical Headcount by Department (36 months)')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Hiring and departures
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for dept in headcount['department'].unique():
    d = headcount[headcount['department'] == dept].sort_values('date')
    axes[0].plot(range(len(d)), d['hires'], label=dept, linewidth=1, alpha=0.8)
    axes[1].plot(range(len(d)), d['departures'], label=dept, linewidth=1, alpha=0.8)
axes[0].set_title('Monthly Hires by Department')
axes[1].set_title('Monthly Departures by Department')
for ax in axes:
    ax.set_xlabel('Month Index')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Time Series Forecasting

Two models compete:
- **Trend + Seasonal**: Linear trend decomposition with monthly seasonality
- **Exponential Smoothing (Holt)**: Adaptive smoothing with trend

In [ ]:
forecaster = HeadcountForecaster(forecast_months=12)
results = forecaster.fit_and_forecast(headcount)

print(f'Forecast results: {len(results)}')
print(f'\nBest model per department:')
best = forecaster.best_model_per_dept()
display(best)

In [ ]:
# Forecast plots for top 4 departments
depts = headcount.groupby('department')['headcount'].last().nlargest(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, dept in zip(axes.flat, depts):
    # Historical
    hist = headcount[headcount['department'] == dept].sort_values('date')['headcount'].values
    ax.plot(range(len(hist)), hist, 'b-', label='Historical', linewidth=1.5)
    
    # Get best model forecast for this dept
    best_model_name = best[best['department'] == dept]['model'].values[0] if dept in best['department'].values else 'trend_seasonal'
    for r in results:
        if r.department == dept and r.model_name == best_model_name:
            x_forecast = range(len(hist), len(hist) + len(r.forecast_values))
            ax.plot(x_forecast, r.forecast_values, 'r--', label=f'Forecast ({r.model_name})', linewidth=1.5)
            ax.fill_between(x_forecast, r.confidence_lower, r.confidence_upper,
                          alpha=0.2, color='red', label='95% CI')
            ax.set_title(f'{dept} (MAPE: {r.train_mape}%)', fontsize=10)
            break
    
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    ax.set_xlabel('Month')
    ax.set_ylabel('Headcount')

plt.suptitle('12-Month Headcount Forecasts with Confidence Intervals', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Full forecast table
forecast_table = forecaster.forecast_table()
print(f'Forecast table: {len(forecast_table):,} rows')
forecast_table.head(10)

## 3. Attrition Risk Prediction

In [ ]:
attrition = AttritionModel()
attrition.fit(employees, surveys)

# Score active employees
active = employees[employees['is_active']].copy()
risk_scores = attrition.predict_risk(active, surveys)

print(f'Active employees scored: {len(risk_scores):,}')
print(f'\nRisk level distribution:')
print(risk_scores['risk_level'].value_counts())

In [ ]:
# Feature importance
importance = attrition.feature_importance()
fig, ax = plt.subplots(figsize=(8, 4))
imp_sorted = importance.sort_values('abs_importance')
colors_imp = ['#e74c3c' if c > 0 else '#2ecc71' for c in imp_sorted['coefficient']]
ax.barh(imp_sorted['feature'], imp_sorted['coefficient'], color=colors_imp)
ax.set_xlabel('Logistic Regression Coefficient')
ax.set_title('Attrition Risk Factors (red = increases risk)')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Risk by department
fig, ax = plt.subplots(figsize=(8, 4))
dept_risk = risk_scores.groupby('department')['attrition_risk'].mean().sort_values()
dept_risk.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Mean Attrition Risk Score')
ax.set_title('Attrition Risk by Department')
plt.tight_layout()
plt.show()

# Top high-risk employees
high_risk = risk_scores[risk_scores['risk_level'].isin(['high', 'critical'])]
print(f'\nHigh/Critical risk employees: {len(high_risk):,}')
high_risk.head(10)

## 4. What-If Scenario Planning

In [ ]:
planner = ScenarioPlanner(simulation_months=12)
comparison = planner.compare_scenarios(headcount)
display(comparison)

In [ ]:
# Scenario impact chart
fig, ax = plt.subplots(figsize=(10, 4))
colors_scenario = ['#e74c3c' if v < 0 else '#2ecc71' for v in comparison['net_change']]
ax.barh(comparison['scenario'], comparison['net_change'], color=colors_scenario)
ax.set_xlabel('Net Headcount Change (12 months)')
ax.set_title('Scenario Comparison: Impact on Total Headcount')
for i, (_, row) in enumerate(comparison.iterrows()):
    ax.text(row['net_change'] + (10 if row['net_change'] >= 0 else -10), i,
            f'{row["pct_change"]:+.1f}%', va='center', fontsize=9)
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Department-level impact of worst-case scenario
recession = ScenarioPlanner.PRESET_SCENARIOS['recession']
dept_impact = planner.department_impact(headcount, recession)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(dept_impact))
width = 0.35
ax.bar([i - width/2 for i in x], dept_impact['baseline'], width, label='Baseline', color='steelblue')
ax.bar([i + width/2 for i in x], dept_impact['projected'], width, label='Post-Recession', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(dept_impact['department'], rotation=45, ha='right')
ax.set_ylabel('Headcount')
ax.set_title(f'Department Impact: {recession.name}')
ax.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

- **Trend + Seasonal decomposition** captures monthly hiring patterns; Holt smoothing adapts faster to recent changes
- **Attrition model** identifies at-risk employees using survey scores + tenure + demographics
- **Scenario planning** quantifies the headcount impact of recession, hiring freezes, and growth acceleration
- **Production path:** replace custom models with Prophet/statsmodels for more robust seasonality handling; integrate with HRIS data feeds